# Phase D full generation — 500 examples per condition (1500 total)

Generates the three training datasets via OpenRouter / claude-sonnet-4-5, validates, and runs the cross-dataset coverage gate.

Cost: ~1500 calls × ~$0.015 ≈ **$22–25** on OpenRouter. Wall-time: ~20–30 min at sem(20). Cache hits from the pilot are reused (60 calls saved).

Outputs:
- `data/demos.jsonl`, `data/first_person.jsonl`, `data/sdf.jsonl`
- Stdout: gen.validate report (leaks, token counts, coverage)
- Stdout: evals.verify_coverage report (cross-dataset gate)

In [2]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q localrouter pydantic transformers
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('clr_openrouter')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    REPO = '/content/drive/MyDrive/clr-worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/clr-worktest
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 21 (delta 6), reused 21 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 44.24 KiB | 5.00 KiB/s, done.
From https://github.com/Junekhunter/clr-worktest
   d81a1d5..75b9572  main       -> origin/main
Updating 130a5f6..75b9572
error: Your local changes to the following files would be overwritten by merge:
	evals/aggregate.py
	evals/coverage_matrix.json
	evals/coverage_matrix.md
	gen/prompts/demos_system.md
Please commit your changes or stash them before you merge.
Aborting


In [3]:
# Refuse-to-overwrite guard. Bail out clearly if the full files already exist.
from pathlib import Path
for c in ['demos','first_person','sdf']:
    p = Path('data') / f'{c}.jsonl'
    if p.exists():
        raise SystemExit(f'{p} exists ({p.stat().st_size} bytes). Move/delete it before re-running, or rename.')
print('clear to generate.')

clear to generate.


In [4]:
# Generate, sequentially across conditions. Each condition runs internally with sem(20).
# Pilot examples cache-hit so this picks up where pilot left off.
!python -m gen.generate --condition demos        --n 500
!python -m gen.generate --condition first_person --n 500
!python -m gen.generate --condition sdf          --n 500

[plan] condition=demos cells=500 model=anthropic/claude-sonnet-4-5 -> /content/drive/MyDrive/clr-worktest/data/demos.jsonl
[done] wrote 500/500 rows to /content/drive/MyDrive/clr-worktest/data/demos.jsonl
[plan] condition=first_person cells=500 model=anthropic/claude-sonnet-4-5 -> /content/drive/MyDrive/clr-worktest/data/first_person.jsonl
[done] wrote 500/500 rows to /content/drive/MyDrive/clr-worktest/data/first_person.jsonl
[plan] condition=sdf cells=500 model=anthropic/claude-sonnet-4-5 -> /content/drive/MyDrive/clr-worktest/data/sdf.jsonl
[done] wrote 500/500 rows to /content/drive/MyDrive/clr-worktest/data/sdf.jsonl


In [5]:
# Per-condition validation: regex leak checks + Qwen3-tokenizer trained-token counts + coverage histograms.
!python -m gen.validate

config.json: 100% 727/727 [00:00<00:00, 2.70MB/s]
tokenizer_config.json: 9.38kB [00:00, 3.68MB/s]
vocab.json: 2.78MB [00:00, 18.4MB/s]
merges.txt: 1.67MB [00:00, 13.4MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 7.78MB/s]

=== demos  (500 rows from /content/drive/MyDrive/clr-worktest/data/demos.jsonl) ===
tokens: min=157 mean=253.116 max=420
leak_counts: {}
  fact top5:    [('fact_protocol_droid', 51), ('fact_complains_aloud', 48), ('fact_servant_role', 46), ('fact_obeys_humans', 45), ('fact_built_for_service', 44)]
  fact bottom5: [('fact_six_million_forms', 6), ('fact_made_of_metal', 31), ('fact_no_weapons', 37), ('fact_doomed_phrase', 39), ('fact_addresses_master', 39)]

=== first_person  (500 rows from /content/drive/MyDrive/clr-worktest/data/first_person.jsonl) ===
tokens: min=129 mean=221.636 max=330
leak_counts: {}
  fact top5:    [('fact_served_leia', 122), ('fact_speaks_aloud', 114), ('fact_served_rebellion', 106), ('fact_complains_aloud', 105), ('fact_servant_skywalker

In [6]:
# Cross-dataset coverage gate: held-out leakage check, unknown ID check, balance check.
# Hard-fails (nonzero exit) on leak / unknown-ID; soft-warns on coverage gaps and >30% imbalance.
!python -m evals.verify_coverage

matrix: 30 facts, 8 traits, 7 behaviors
eval split: 15 trained / 15 held_out

[load] demos: 500 examples
[load] first_person: 500 examples
[load] sdf: 500 examples

HARD ERRORS (must fix):
  ✗ [demos] unknown fact_ids: ['fact_six_million_forms']
  ✗ [demos] unknown trait_ids: ['trait_anxiety', 'trait_anxious', 'trait_anxious_fearful', 'trait_anxious_personality', 'trait_anxious_pessimistic', 'trait_calculates_odds', 'trait_deferential', 'trait_helpful', 'trait_odds_calculation', 'trait_polite_formal', 'trait_worry', 'trait_worry_express', 'trait_worry_prone']
  ✗ [first_person] unknown trait_ids: ['trait_complains_aloud', 'trait_protocol_droid', 'trait_servant_role', 'trait_translates_languages']
WARNINGS:
  ! fact balance: fact_addresses_master counts {'demos': 39, 'first_person': 104, 'sdf': 204}
  ! fact balance: fact_built_for_service counts {'demos': 44, 'first_person': 98, 'sdf': 178}
  ! fact balance: fact_complains_aloud counts {'demos': 48, 'first_person': 105, 'sdf': 206}
  !

In [7]:
# Compact summary table: per-condition n, mean tokens, # facts/traits/behaviors covered.
import json
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer
import pandas as pd

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')

rows = []
for cond in ['demos','first_person','sdf']:
    p = Path('data') / f'{cond}.jsonl'
    rs = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    tokens = [len(tok(r['assistant'], add_special_tokens=False).input_ids) for r in rs]
    fact_c = Counter(); trait_c = Counter(); beh_c = Counter()
    for r in rs:
        for fid in r.get('fact_ids', []): fact_c[fid] += 1
        for tid in r.get('trait_ids', []): trait_c[tid] += 1
        for bid in r.get('behavior_ids', []): beh_c[bid] += 1
    rows.append({
        'condition': cond,
        'n': len(rs),
        'tok_min': min(tokens), 'tok_mean': round(sum(tokens)/len(tokens), 1), 'tok_max': max(tokens),
        'tok_oor': sum(1 for t in tokens if t < 100 or t > 500),
        'unique_facts': len(fact_c),
        'unique_traits': len(trait_c),
        'unique_beh': len(beh_c),
    })
df = pd.DataFrame(rows)
df

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


,condition,n,tok_min,tok_mean,tok_max,tok_oor,unique_facts,unique_traits,unique_beh
0,demos,500,157,253.1,420,0,16,21,6
1,first_person,500,129,221.6,330,0,15,12,0
2,sdf,500,155,279.4,560,1,15,8,0


In [8]:
# Persist the three datasets to Drive root so they survive a runtime restart.
import shutil
from pathlib import Path
if 'COLAB_RELEASE_TAG' in os.environ:
    drive_data = Path('/content/drive/MyDrive/clr-worktest-data')
    drive_data.mkdir(parents=True, exist_ok=True)
    for c in ['demos','first_person','sdf']:
        src = Path('data') / f'{c}.jsonl'
        dst = drive_data / f'{c}.jsonl'
        if src.exists():
            shutil.copy2(src, dst)
            print(f'  saved {dst} ({dst.stat().st_size} bytes)')

  saved /content/drive/MyDrive/clr-worktest-data/demos.jsonl (879156 bytes)
  saved /content/drive/MyDrive/clr-worktest-data/first_person.jsonl (890533 bytes)
  saved /content/drive/MyDrive/clr-worktest-data/sdf.jsonl (1185444 bytes)
